# TNG X-ray Pipeline — Showcase

This notebook demonstrates the intrinsic X-ray pipeline end-to-end:

1. **Single halo** — run the pipeline on one cluster, inspect the output
2. **Batch validation** — load pre-computed 30-halo results, compare to TNG reference projections
3. **Systematic analysis** — ΔlogLx vs redshift, empirical correction

---
**Pipeline summary:** per-particle VAPEC emissivity (APEC v3.0.9, Asplund+2009),  
SPH-kernel projection with `sphviewer2`, 4×R200c FOV, ±R200c depth.  
Loader: `zoom` (TNG-Cluster default).  Band: 0.5–5 keV rest-frame, zobs=0.

## 0. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.optimize import curve_fit

# Pipeline
from xray_pipeline import process_halo, SIM_CONFIGS

# Paths
RESULTS_HDF5 = 'results/results.hdf5'          # from run_sample.py
PROJ_DIR     = ('/virgotng/mpia/TNG-Cluster/L680n8192TNG'
                '/postprocessing/projections')
PROJ_TMPL    = os.path.join(PROJ_DIR,
               'gas-xray_lum_0.5-5.0kev__2r200_d=r200.{snap}.hdf5')

VIEW    = 0        # xy-plane projection
N_PIX   = 2000
FOV     = 4.0      # units of R200c

print('Setup complete.')

---
## 1. Single-halo demo

Run the full pipeline on one cluster (snap 99, z=0, halo_id=0 — the most massive TNG-Cluster zoom target).
This takes ~60–90 s with 8 threads.

In [ ]:
cfg = SIM_CONFIGS['tng-cluster']

result = process_halo(
    halo_id   = 0,
    snap      = 99,
    base_path = cfg.base_path,
    loader    = cfg.loader,      # 'zoom'
    sim       = 'tng-cluster',
    nthreads  = 8,
)

meta = result.meta
print(f'z          = {1/meta.a - 1:.4f}')
print(f'M200c      = {meta.M200c_msun:.3e} M_sun')
print(f'R200c      = {meta.R200c_kpc:.1f} kpc')
print(f'L_x(R500c) = {meta.L_x_r500c:.3e} erg/s   (log10 = {np.log10(meta.L_x_r500c):.2f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
titles = ['xy plane  (view 0)', 'xz plane  (view 1)', 'yz plane  (view 2)']

for v, ax in enumerate(axes):
    img = result.images[:, :, v]
    valid = img[img > -99]
    vmax  = float(np.percentile(valid, 99.5)) if len(valid) else 0
    ax.imshow(img, cmap='inferno', vmin=vmax - 4, vmax=vmax, origin='lower')
    ax.set_title(titles[v], fontsize=9)
    ax.axis('off')

fig.suptitle(f'TNG-Cluster halo 0  |  snap 99  |  z=0  |  0.5–5 keV  |  log₁₀ SB [erg/s/kpc²]',
             fontsize=9)
fig.tight_layout()
plt.show()

---
## 2. Load pre-computed 30-halo results

Results from `run_sample.py --n 30 --seed 42` — 30 halos sampled uniformly across  
all 13 available reference snapshots (z = 0 → 5).

In [ ]:
def lx_from_image(log_sb, r200c_kpc):
    """Integrate log10-SB image to total Lx [erg/s] using physical pixel area."""
    kpp = FOV * r200c_kpc / N_PIX
    sb  = np.where(log_sb <= -99.0, 0.0, 10.0 ** log_sb)
    return float(sb.sum() * kpp ** 2)


rows = []
ref_cache = {}

with h5py.File(RESULTS_HDF5, 'r') as fh:
    for key in sorted(fh['halos'].keys()):
        grp   = fh['halos'][key]
        attrs = dict(grp.attrs)
        if attrs.get('failed', False):
            continue

        snap     = int(attrs['snap'])
        halo_id  = int(attrs['halo_id'])
        zoom_idx = int(attrs['zoom_idx'])
        a        = float(attrs['a'])
        z        = round(1.0 / a - 1.0, 3)
        r200c    = float(attrs['R200c_kpc'])
        lx_r500c = float(attrs['L_x_r500c'])

        pipe_img = grp['images'][:, :, VIEW].astype(np.float64)

        # Load matching reference projection
        if snap not in ref_cache:
            ref_cache[snap] = h5py.File(PROJ_TMPL.format(snap=snap), 'r')
        ref_img = ref_cache[snap][f'Halo_{halo_id}'][:, :, VIEW].astype(np.float64)

        lx_ref  = lx_from_image(ref_img,  r200c)
        lx_pipe = lx_from_image(pipe_img, r200c)
        dlx     = (np.log10(lx_pipe) - np.log10(lx_ref)
                   if lx_ref > 0 and lx_pipe > 0 else float('nan'))

        rows.append(dict(
            key=key, snap=snap, halo_id=halo_id, zoom_idx=zoom_idx,
            z=z, a=a, r200c=r200c, lx_r500c=lx_r500c,
            lx_ref=lx_ref, lx_pipe=lx_pipe, dlx=dlx,
            ref_img=ref_img, pipe_img=pipe_img,
        ))

for fh in ref_cache.values():
    fh.close()

rows.sort(key=lambda r: (-r['z'], -r['lx_r500c']))
print(f'{len(rows)} halos loaded')

# Summary table
print(f"\n{'i':>3}  {'snap':>4}  {'z':>5}  {'zoom':>4}  {'logLx_pipe':>10}  {'logLx_ref':>9}  {'ΔlogLx':>7}")
print('-' * 65)
for i, r in enumerate(rows):
    lp = np.log10(r['lx_pipe']) if r['lx_pipe'] > 0 else float('nan')
    lr = np.log10(r['lx_ref'])  if r['lx_ref']  > 0 else float('nan')
    d  = f"{r['dlx']:+.3f}" if np.isfinite(r['dlx']) else '  NaN'
    print(f"{i:3d}  {r['snap']:4d}  {r['z']:5.3f}  {r['zoom_idx']:4d}  "
          f"{lp:10.3f}  {lr:9.3f}  {d:>7}")

---
## 3. Validation figure: pipeline vs TNG reference

30 rows × 3 columns: Reference (TNG) | Pipeline (VAPEC, zobs=0) | Residual (ref − pipe).  
Rows sorted by redshift (high-z first).

In [ ]:
def _cutoff(log_sb):
    h, w = log_sb.shape
    y, x  = np.ogrid[:h, :w]
    dist  = np.sqrt((y - h/2)**2 + (x - w/2)**2)
    mask  = dist <= np.sqrt(0.1) * (w / 2.0)
    vals  = log_sb[mask]
    vals  = vals[np.isfinite(vals)]
    return float(np.percentile(vals, 99)) if len(vals) else 0.0

def _normalise(log_sb, cutoff):
    arr = np.where(log_sb <= -99.0, np.nan, log_sb)
    n   = (arr - (cutoff - 4.0)) / 4.0
    return np.nan_to_num(np.clip(n, 0.0, 1.0), nan=0.0)


N     = len(rows)
ROW_H = 1.5
fig, axes = plt.subplots(N, 3,
    figsize=(11, N * ROW_H),
    gridspec_kw={'wspace': 0.03, 'hspace': 0.35},
)

axes[0, 0].set_title('Reference (TNG)',             fontsize=7, pad=3)
axes[0, 1].set_title('Pipeline (VAPEC, zobs=0)',     fontsize=7, pad=3)
axes[0, 2].set_title('Residual  ref − pipe  [dex]', fontsize=7, pad=3)

for ri, r in enumerate(rows):
    ax_ref, ax_pipe, ax_res = axes[ri]

    cutoff = _cutoff(r['ref_img'])
    ax_ref.imshow(_normalise(r['ref_img'],  cutoff), cmap='inferno',
                  vmin=0, vmax=1, origin='lower')
    ax_pipe.imshow(_normalise(r['pipe_img'], cutoff), cmap='inferno',
                   vmin=0, vmax=1, origin='lower')

    ref_c  = np.where(r['ref_img']  <= -99.0, np.nan, r['ref_img'])
    pipe_c = np.where(r['pipe_img'] <= -99.0, np.nan, r['pipe_img'])
    resid  = ref_c - pipe_c
    abs_max = max(float(np.nanpercentile(np.abs(resid), 99)), 0.05)
    norm   = TwoSlopeNorm(vcenter=0, vmin=-abs_max, vmax=abs_max)
    im     = ax_res.imshow(resid, cmap='RdBu_r', norm=norm, origin='lower')
    plt.colorbar(im, ax=ax_res, fraction=0.046, pad=0.01)

    d_str = f"Δ={r['dlx']:+.2f}" if np.isfinite(r['dlx']) else 'Δ=N/A'
    ax_res.text(0.02, 0.97, d_str, transform=ax_res.transAxes,
                ha='left', va='top', fontsize=5.5, color='white',
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6, lw=0))

    ax_ref.set_ylabel(f"snap{r['snap']}\nz={r['z']:.2f}\nzoom{r['zoom_idx']}",
                      fontsize=5.5, rotation=0, labelpad=42, va='center')

    for ax in (ax_ref, ax_pipe, ax_res):
        ax.axis('off')

finite_dlx = [r['dlx'] for r in rows if np.isfinite(r['dlx'])]
med = np.median(finite_dlx)
rms = np.sqrt(np.mean(np.array(finite_dlx)**2))
fig.suptitle(
    f'30-halo sample  |  VAPEC zobs=0  |  0.5–5 keV  |  '
    f'median Δ={med:+.3f} dex   rms={rms:.3f} dex',
    fontsize=8, y=1.002,
)

fig.savefig('results/validation_30halos.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved → results/validation_30halos.png')

---
## 4. ΔlogLx vs redshift

The pipeline overestimates Lx relative to the TNG reference, with the offset growing  
smoothly with redshift.  Best empirical fit: Δ ≈ 1.26 × log₁₀(1+z).

In [ ]:
zs  = np.array([r['z']   for r in rows if np.isfinite(r['dlx'])])
dlx = np.array([r['dlx'] for r in rows if np.isfinite(r['dlx'])])
as_ = 1.0 / (1.0 + zs)

# Fit: Δ = n * log10(1+z)
def model(z, n):
    return n * np.log10(1.0 + z)

popt, _ = curve_fit(model, zs, dlx)
n_best  = float(popt[0])
dlx_fit = model(zs, n_best)
rms_fit = np.sqrt(np.mean((dlx - dlx_fit)**2))

z_grid  = np.linspace(0, zs.max() * 1.05, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── left: Δ vs z ──
ax = axes[0]
ax.scatter(zs, dlx, c='steelblue', zorder=3, label='30 halos')
ax.plot(z_grid, model(z_grid, n_best), 'r-',
        label=f'best fit: Δ = {n_best:.2f}·log₁₀(1+z)  (rms={rms_fit:.3f})')
ax.plot(z_grid, model(z_grid, 1.0),   'k--', alpha=0.5, label='n=1.0 (1/a)')
ax.plot(z_grid, model(z_grid, 2.0),   'k:',  alpha=0.5, label='n=2.0 (1/a²)')
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('Redshift z', fontsize=11)
ax.set_ylabel('Δ log₁₀ Lx  (pipe − ref)', fontsize=11)
ax.set_title('Pipeline vs TNG reference: Lx offset', fontsize=11)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── right: per-snapshot box whisker ──
ax2 = axes[1]
snaps_unique = sorted(set(r['snap'] for r in rows if np.isfinite(r['dlx'])))
snap_to_z    = {r['snap']: r['z'] for r in rows}

positions = []
data_snap  = []
for s in snaps_unique:
    vals = [r['dlx'] for r in rows if r['snap'] == s and np.isfinite(r['dlx'])]
    if vals:
        data_snap.append(vals)
        positions.append(snap_to_z[s])

bp = ax2.boxplot(data_snap, positions=positions, widths=0.08,
                  patch_artist=True, manage_ticks=False,
                  boxprops=dict(facecolor='lightsteelblue'),
                  medianprops=dict(color='navy', lw=2))
ax2.plot(z_grid, model(z_grid, n_best), 'r-',
         label=f'n={n_best:.2f} fit')
ax2.axhline(0, color='gray', lw=0.8)
ax2.set_xlabel('Redshift z', fontsize=11)
ax2.set_ylabel('Δ log₁₀ Lx  (pipe − ref)', fontsize=11)
ax2.set_title('Per-snapshot distribution', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('results/dlx_vs_redshift.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best-fit n = {n_best:.3f}  (rms = {rms_fit:.3f} dex)')
print(f'Saved → results/dlx_vs_redshift.png')

---
## 5. Known systematic — status summary

| Hypothesis | Tested? | Outcome |
|---|---|---|
| K-correction (zobs = z) | ✓ | Makes it worse: −1.7 dex at z=5 |
| GFM_CoolingRate sign error | ✓ | Confirmed correct (dU/dt, negative=cooling) |
| T_min threshold (3e5 → 1e6 K) | ✓ | Minor effect, does not explain trend |
| APEC table version | ✓ | No obvious mismatch |
| FoF vs zoom loader | ✓ | ~0.03 dex difference, not the cause |
| Comoving vs physical kpc² in reference | ✓ | r200c and kpp match exactly |
| a from header vs computed from z | ✓ | Already reading from header |
| **APEC version of TNG reference** | ✗ | **Leading untested hypothesis** |

**Next step:** contact the TNG-Cluster postprocessing team to identify the APEC  
version used to generate `gas-xray_lum_0.5-5.0kev__2r200_d=r200.{snap}.hdf5`.  
At z > 1, clusters are cooler (1–3 keV) and soft X-ray line emission dominates;  
APEC versions differ significantly in that regime.